# 04 - Business Impact & Threshold Optimization

Find optimal prediction threshold using cost-benefit analysis

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_recall_curve
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# FIND THE FILE - ABSOLUTE METHOD
# ============================================================

print("Current working directory:", os.getcwd())
print("\nSearching for telco_churn.csv...\n")

# Search for the file in your home directory
import subprocess
result = subprocess.run(['find', os.path.expanduser('~'), '-name', 'telco_churn.csv', '-type', 'f'], 
                       capture_output=True, text=True, timeout=10)

files_found = result.stdout.strip().split('\n')
files_found = [f for f in files_found if f]

if files_found:
    print(f"Found {len(files_found)} file(s):")
    for i, file in enumerate(files_found[:5], 1):
        print(f"  {i}. {file}")
    
    data_path = files_found[0]
    print(f"\n✅ Using: {data_path}\n")
else:
    print("❌ File not found. Please enter the full path:")
    data_path = input("Path: ").strip()

# ============================================================
# LOAD DATA
# ============================================================

try:
    df = pd.read_csv(data_path)
    print(f'✅ Data loaded: {df.shape}\n')
except Exception as e:
    print(f"❌ Error: {e}")
    exit()

# CLEAN DATA
df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# ENCODE
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# SCALE
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# SPLIT
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}\n')

# ============================================================
# TRAIN MODELS
# ============================================================

print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f'✅ AUC: {lr_auc:.4f}')

print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f'✅ AUC: {rf_auc:.4f}')

print('Training XGBoost...')
xgb = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
print(f'✅ AUC: {xgb_auc:.4f}\n')

# ============================================================
# THRESHOLD OPTIMIZATION
# ============================================================

print('='*70)
print('THRESHOLD OPTIMIZATION & BUSINESS IMPACT')
print('='*70)

y_proba = xgb.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

best_cost = float('inf')
best_threshold = 0.5
best_tp = best_fp = best_fn = best_tn = 0

for threshold in thresholds:
    preds = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    cost = (fp * 500) + (fn * 5000)
    
    if cost < best_cost:
        best_cost = cost
        best_threshold = threshold
        best_tp, best_fp, best_fn, best_tn = tp, fp, fn, tn

print(f'\n🎯 OPTIMAL THRESHOLD: {best_threshold:.3f}')
print(f'\n📊 CONFUSION MATRIX:')
print(f'   TP: {best_tp}  | FP: {best_fp}')
print(f'   FN: {best_fn}  | TN: {best_tn}')

print(f'\n💰 ANNUAL IMPACT:')
annual_saved = best_tp * 12
annual_value = (best_tp * 5000) * 12
annual_spend = (best_tp * 500) * 12
annual_benefit = annual_value - annual_spend

print(f'   Saved: {annual_saved} customers/year')
print(f'   Value: ${annual_value:,}/year')
print(f'   Spend: ${annual_spend:,}/year')
print(f'   BENEFIT: ${annual_benefit:,}/year')

print(f'\n' + '='*70)
print('✅ PROJECT COMPLETE!')
print('='*70)
print(f'\nBest Model: XGBoost (AUC: {xgb_auc:.4f})')
print(f'Optimal Threshold: {best_threshold:.3f}')
print(f'Annual Benefit: ${annual_benefit:,}')
print(f'Status: BUSINESS READY ✅')

# PLOT
plt.figure(figsize=(10, 6))
costs_list = []
for threshold in thresholds:
    preds = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    cost = (fp * 500) + (fn * 5000)
    costs_list.append(cost)

plt.plot(thresholds, costs_list, linewidth=2, color='#3498db')
plt.axvline(x=best_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal: {best_threshold:.3f}')
plt.xlabel('Prediction Threshold')
plt.ylabel('Total Cost ($)')
plt.title('Cost vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Current working directory: /Users/koutilyayenumula/churn-prediction/notebooks

Searching for telco_churn.csv...



TimeoutExpired: Command '['find', '/Users/koutilyayenumula', '-name', 'telco_churn.csv', '-type', 'f']' timed out after 10 seconds